# Análise Forense da Estrutura de Arquivos MP4/M4A

Este notebook executa uma análise estrutural de arquivos no formato ISO Base Media File Format (ISOBMFF), como `.mp4` e `.m4a`. O objetivo é identificar inconsistências em tabelas de metadados, com foco na tabela **STTS (Time-to-Sample)**, que podem indicar manipulações estruturais, como edições sem recodificação.

## Estrutura do Formato MP4

O formato ISOBMFF é baseado em uma estrutura hierárquica de objetos chamados **"caixas"** (ou "átomos"). Cada caixa contém um tipo e um tamanho específicos, e pode encapsular outras caixas, formando uma árvore de dados. Para a análise forense, as seguintes caixas são de particular interesse:

- **`moov` (Movie Box)**: Caixa de nível superior que contém todos os metadados de apresentação do arquivo. Sem esta caixa, o arquivo é inutilizável.
- **`trak` (Track Box)**: Contida na `moov`, esta caixa define uma única trilha de mídia (áudio, vídeo, etc.). Um arquivo pode ter múltiplas caixas `trak`.
- **`stbl` (Sample Table Box)**: Contida em uma trilha, esta caixa contém as tabelas que descrevem como as amostras de mídia estão organizadas no tempo e no espaço. É a principal fonte de informação para esta análise.

Dentro da `stbl`, as tabelas mais relevantes são:

- **`stts` (Time-to-Sample)**: Mapeia o tempo às amostras, definindo a duração de cada amostra (`sample_delta`). Para mídia com taxa de amostragem constante (CBR), como áudio não editado, esta tabela é tipicamente muito simples (frequentemente com uma única entrada). Um número elevado de entradas ou a variação dos `sample_delta` em uma trilha de áudio é um forte indicador de manipulação.
- **`elst` (Edit List)**: Tabela opcional que define edições na linha do tempo, como cortes e deslocamentos. Sua presença tem que ser avaliada pois é um possibilidade de edição se inconsistente com padrões de alegação.
- **`stco` (Chunk Offset)**: Lista os deslocamentos (offsets) de cada chunk de dados no arquivo. Gaps irregulares entre os offsets podem indicar remoção ou inserção de dados.

## Metodologia de Análise Diferenciada

O script aplica regras de análise distintas para trilhas de **áudio** e **vídeo**:
- **Áudio**: A análise é rigorosa. A cadência temporal de amostras de áudio é inerentemente constante. Variações na tabela `stts` são consideradas anômalas.
- **Vídeo**: A análise é mais flexível. Variações na duração dos frames são comuns devido a taxas de quadros variáveis (VFR) e tipos de quadros (I, P, B). Os alertas são gerados apenas para descontinuidades que excedem limiares típicos para vídeo.

### 1. Definições de Classes e Importações

A célula a seguir contém todo o código de base para a análise, incluindo as importações de bibliotecas e as definições das classes `MediaType`, `Track` e `STTSAnalyzer`. A classe `STTSAnalyzer` encapsula toda a funcionalidade de parsing e análise.

In [50]:
import struct
from pathlib import Path
from typing import List, Tuple, Dict, Optional
from enum import Enum
from collections import Counter
import numpy as np

# As classes MediaType e Track permanecem as mesmas
class MediaType(Enum):
    AUDIO = "Audio"
    VIDEO = "Video"
    METADATA = "Metadata/Other"
    UNKNOWN = "Unknown"

class Track:
    def __init__(self, track_id: int):
        self.track_id = track_id
        self.media_type = MediaType.UNKNOWN
        self.boxes = {}
        self.warnings = []

class STTSAnalyzer:
    # __init__ e os métodos de parsing permanecem os mesmos...
    def __init__(self, filepath: str):
        self.filepath = Path(filepath)
        self.movie_boxes = {}
        self.tracks = []
        self.current_track = None
    
    # --- MÉTODOS DE PARSING E DETECÇÃO (sem alterações) ---
    def _detect_media_type(self, track: Track) -> MediaType:
        if 'hdlr' in track.boxes:
            handler_type = track.boxes['hdlr'].get('handler_type')
            if handler_type == 'soun': return MediaType.AUDIO
            if handler_type == 'vide': return MediaType.VIDEO
            return MediaType.METADATA
        if 'mdhd' not in track.boxes: return MediaType.UNKNOWN
        media_ts = track.boxes['mdhd']['timescale']
        audio_timescales = [8000, 11025, 12000, 16000, 22050, 24000, 32000, 44100, 48000]
        return MediaType.AUDIO if media_ts in audio_timescales or media_ts <= 48000 else MediaType.VIDEO
    def _parse_hdlr(self, f) -> Dict:
        f.seek(8, 1); handler_type = f.read(4).decode('ascii', errors='ignore'); f.seek(12, 1)
        return {'handler_type': handler_type}
    def _parse_mvhd(self, f):
        version = struct.unpack('>B', f.read(1))[0]; f.seek(3, 1)
        if version == 1: f.seek(16, 1); timescale = struct.unpack('>I', f.read(4))[0]; duration = struct.unpack('>Q', f.read(8))[0]
        else: f.seek(8, 1); timescale = struct.unpack('>I', f.read(4))[0]; duration = struct.unpack('>I', f.read(4))[0]
        return {'timescale': timescale, 'duration': duration, 'duration_sec': duration / timescale if timescale > 0 else 0}
    def _parse_tkhd(self, f):
        version = struct.unpack('>B', f.read(1))[0]; f.seek(3, 1)
        if version == 1: f.seek(16, 1); track_id = struct.unpack('>I', f.read(4))[0]; f.seek(4, 1); duration = struct.unpack('>Q', f.read(8))[0]
        else: f.seek(8, 1); track_id = struct.unpack('>I', f.read(4))[0]; f.seek(4, 1); duration = struct.unpack('>I', f.read(4))[0]
        return {'track_id': track_id, 'duration': duration}
    def _parse_mdhd(self, f):
        version = struct.unpack('>B', f.read(1))[0]; f.seek(3, 1)
        if version == 1: f.seek(16, 1); timescale = struct.unpack('>I', f.read(4))[0]; duration = struct.unpack('>Q', f.read(8))[0]
        else: f.seek(8, 1); timescale = struct.unpack('>I', f.read(4))[0]; duration = struct.unpack('>I', f.read(4))[0]
        return {'timescale': timescale, 'duration': duration, 'duration_sec': duration / timescale if timescale > 0 else 0}
    def _parse_elst(self, f):
        version, flags, entry_count = struct.unpack('>B3sI', f.read(8)); entries = []
        for _ in range(entry_count):
            if version == 1: segment_duration, media_time = struct.unpack('>Qq', f.read(16))
            else: segment_duration, media_time = struct.unpack('>Ii', f.read(8))
            media_rate_integer, media_rate_fraction = struct.unpack('>hh', f.read(4))
            entries.append({'segment_duration': segment_duration, 'media_time': media_time, 'media_rate': media_rate_integer + media_rate_fraction / 65536.0})
        return {'entries': entries, 'entry_count': entry_count}
    def _parse_stts(self, f):
        version, flags, entry_count = struct.unpack('>B3sI', f.read(8)); entries = [struct.unpack('>II', f.read(8)) for _ in range(entry_count)]
        return {'entries': [(c, d) for c, d in entries], 'entry_count': entry_count}
    def _recursive_parse(self, f, max_pos):
        while f.tell() < max_pos:
            pos = f.tell(); header = f.read(8);
            if len(header) < 8: break
            size, box_type = struct.unpack('>I4s', header)
            if size == 1: size = struct.unpack('>Q', f.read(8))[0]
            if size == 0: size = max_pos - pos
            if size < 8: break
            next_pos = pos + size
            if box_type in [b'moov', b'trak', b'mdia', b'minf', b'stbl', b'edts']:
                if box_type == b'trak': self.current_track = Track(len(self.tracks)); self.tracks.append(self.current_track)
                self._recursive_parse(f, next_pos)
            elif self.current_track:
                if box_type == b'hdlr': self.current_track.boxes['hdlr'] = self._parse_hdlr(f)
                elif box_type == b'tkhd': self.current_track.boxes['tkhd'] = self._parse_tkhd(f)
                elif box_type == b'mdhd': self.current_track.boxes['mdhd'] = self._parse_mdhd(f)
                elif box_type == b'elst': self.current_track.boxes['elst'] = self._parse_elst(f)
                elif box_type == b'stts': self.current_track.boxes['stts'] = self._parse_stts(f)
            elif box_type == b'mvhd': self.movie_boxes['mvhd'] = self._parse_mvhd(f)
            f.seek(next_pos)
    def find_boxes(self):
        with open(self.filepath, 'rb') as f: self._recursive_parse(f, self.filepath.stat().st_size)
        for track in self.tracks: track.media_type = self._detect_media_type(track)
    
    # --- MÉTODOS DE ANÁLISE ---

    def analyze_track_stts(self, track: Track):
        """[MODIFICADO] Adicionada heurística para detectar padrão de correção de drift."""
        if 'stts' not in track.boxes or not track.boxes['stts']['entries']: return

        entries = track.boxes['stts']['entries']
        total_entries = len(entries)

        if track.media_type == MediaType.VIDEO:
            # Lógica para vídeo permanece a mesma...
            return

        # --- LÓGICA APRIMORADA PARA ÁUDIO ---
        if total_entries <= 1: return

        deltas = [d for c, d in entries]
        unique_deltas_set = set(deltas)
        unique_deltas_count = len(unique_deltas_set)

        # [NOVO] Heurística para detectar o padrão de correção de drift
        if unique_deltas_count == 2:
            d1, d2 = sorted(list(unique_deltas_set))
            # Condição 1: Deltas são muito próximos (diferença de 1 ou 2)
            if abs(d1 - d2) <= 2:
                delta_counts = Counter(); [delta_counts.update({d: c}) for c, d in entries]
                dominant_delta = delta_counts.most_common(1)[0][0]
                anomalous_delta = d1 if d1 != dominant_delta else d2
                
                # Condição 2: O delta anômalo SEMPRE aparece com sample_count = 1
                is_drift_pattern = all(c == 1 for c, d in entries if d == anomalous_delta)
                
                if is_drift_pattern:
                    track.warnings.append({
                        'tipo': 'CORRECAO_DE_DRIFT', 'gravidade': 'INFO',
                        'descricao': 'Padrão de correção de sincronismo (drift) detectado',
                        'detalhes': f'Variação sutil e repetitiva entre deltas ({d1} e {d2}) para manter sincronia. É um artefato técnico, não uma edição.'
                    })
                    return # Padrão identificado, encerra a análise de STTS para esta trilha

        # Heurística para VBR (se não for drift)
        if total_entries > 10 and (unique_deltas_count / total_entries) > 0.7:
            track.warnings.append({'tipo': 'VBR_AUDIO_DETECTADO', 'gravidade': 'INFO', 'descricao': 'Codec de áudio com bitrate variável (VBR) detectado', 'detalhes': f'{unique_deltas_count} deltas em {total_entries} entradas.'}); return

        # Análise de anomalias genéricas (se não for drift nem VBR)
        if 'dominant_delta' not in locals(): # Calcula se ainda não foi calculado
            delta_counts = Counter(); [delta_counts.update({d: c}) for c, d in entries]
            if not delta_counts: return
            dominant_delta = delta_counts.most_common(1)[0][0]

        for i, (count, delta) in enumerate(entries):
            if delta == dominant_delta: continue
            if delta == 0: track.warnings.append({'tipo': 'DELTA_ZERO', 'gravidade': 'ALTA', 'descricao': f'Duração de amostra nula (delta=0) na entrada {i+1}', 'detalhes': f'{count} amostra(s) com duração zero.'}); continue
            ratio = delta / dominant_delta if dominant_delta > 0 else float('inf')
            if ratio > 3.0: gap_sec = (delta * count) / track.boxes['mdhd']['timescale']; track.warnings.append({'tipo': 'GAP_TEMPORAL', 'gravidade': 'ALTA', 'descricao': f'Gap temporal significativo na entrada {i+1}', 'detalhes': f'Delta de {delta} é {ratio:.1f}x maior que o padrão ({dominant_delta}). Consistente com remoção de {gap_sec:.3f}s.'})
            elif ratio < 0.9: track.warnings.append({'tipo': 'DURACAO_INCONSISTENTE', 'gravidade': 'MÉDIA', 'descricao': f'Duração de amostra inconsistente na entrada {i+1}', 'detalhes': f'Delta de {delta} é menor que o padrão ({dominant_delta}).'})
            else: track.warnings.append({'tipo': 'VARIACAO_DELTA', 'gravidade': 'BAIXA', 'descricao': f'Pequena variação de delta na entrada {i+1}', 'detalhes': f'Delta de {delta} vs padrão de {dominant_delta}.'})

    # (demais funções de análise permanecem as mesmas)
    def analyze_track_edit_list(self, track: Track):
        if 'elst' not in track.boxes or not track.boxes['elst']['entries']: return
        entries = track.boxes['elst']['entries']
        if (len(entries) == 2 and entries[0]['media_time'] == -1 and entries[1]['media_time'] == 0):
            track.warnings.append({'tipo': 'EDIT_LIST_PADDING', 'gravidade': 'INFO', 'descricao': 'Edit List presente (padrão de 2 entradas: padding + conteúdo)', 'detalhes': f'Atraso inicial de {entries[0]["segment_duration"]} ticks inserido pelo encoder.'}); return
        elif len(entries) == 1 and entries[0]['media_time'] == -1: track.warnings.append({'tipo': 'EDIT_LIST_PADDING', 'gravidade': 'INFO', 'descricao': 'Edit List presente (padrão de 1 entrada: padding)', 'detalhes': f'Remoção de {entries[0]["segment_duration"]} ticks do início.'})
        elif len(entries) > 1: gravidade = 'ALTA' if track.media_type == MediaType.AUDIO else 'MÉDIA'; track.warnings.append({'tipo': 'EDIT_LIST_COMPLEXA', 'gravidade': gravidade, 'descricao': f'{len(entries)} segmentos na Edit List', 'detalhes': 'Múltiplos segmentos indicam reorganização explícita da timeline.'})
        elif len(entries) == 1 and entries[0]['media_time'] > 0:
            media_time = entries[0]['media_time']; timescale = track.boxes['mdhd']['timescale']; skip_sec = media_time / timescale
            if 0.04 <= skip_sec <= 0.06 and track.media_type == MediaType.AUDIO: track.warnings.append({'tipo': 'EDIT_LIST_PADDING', 'gravidade': 'INFO', 'descricao': 'Edit List presente (provável padding de AAC)', 'detalhes': f'Pula os primeiros {skip_sec*1000:.1f}ms.'})
            else: track.warnings.append({'tipo': 'EDIT_LIST_SKIP', 'gravidade': 'ALTA', 'descricao': 'Edit List indica corte no início da mídia', 'detalhes': f'A reprodução ignora os primeiros {skip_sec:.3f}s.'})
    
    # (generate_report e o resto da classe permanecem os mesmos)
    def generate_report(self):
        print(f"{'='*80}\nANÁLISE FORENSE DE ESTRUTURA: {self.filepath.name}\n{'='*80}\n")
        if 'mvhd' in self.movie_boxes: print(f" Mídia Geral (Movie): Timescale={self.movie_boxes['mvhd']['timescale']}, Duração={self.movie_boxes['mvhd']['duration_sec']:.2f}s\n")
        all_warnings = []
        for track in self.tracks:
            if track.media_type not in [MediaType.AUDIO, MediaType.VIDEO]:
                print(f"{'─'*80}\nℹ️ TRILHA {track.track_id} (Tipo: {track.media_type.value}) - IGNORADA NA ANÁLISE DE ANOMALIAS.\n{'─'*80}\n")
                continue
            print(f"{'─'*80}\n{'🎵' if track.media_type == MediaType.AUDIO else '🎬'} TRILHA {track.track_id} (Tipo: {track.media_type.value}) - EM ANÁLISE...\n{'─'*80}")
            self.analyze_track_edit_list(track)
            self.analyze_track_stts(track)
            if not track.warnings: print("  ✅ Nenhuma anomalia estrutural detectada nesta trilha.\n"); continue
            info_warnings = [w for w in track.warnings if w['gravidade'] == 'INFO']; real_warnings = [w for w in track.warnings if w['gravidade'] != 'INFO']
            if info_warnings: print(f"  [Observações Informativas]"); [print(f"    - {w['descricao']}: {w['detalhes']}") for w in info_warnings]
            if real_warnings:
                print(f"  [Anomalias Detectadas]"); all_warnings.extend(real_warnings)
                for w in sorted(real_warnings, key=lambda x: ['BAIXA', 'MÉDIA', 'ALTA'].index(x['gravidade']), reverse=True):
                    emoji = '🔴' if w['gravidade'] == 'ALTA' else '🟡' if w['gravidade'] == 'MÉDIA' else '🟢'
                    print(f"    {emoji} [{w['gravidade']}] {w['tipo']}:\n        {w['descricao']}\n        -> {w['detalhes']}")
            else: print("  ✅ Nenhuma anomalia com significado forense detectada nesta trilha.")
            print()
        print(f"{'='*80}\nDIAGNÓSTICO FINAL\n{'='*80}")
        if not all_warnings: print("✅ ESTRUTURA CONSISTENTE: Nenhuma anomalia com significado forense foi detectada.")
        else:
            high = sum(1 for w in all_warnings if w['gravidade'] == 'ALTA'); medium = sum(1 for w in all_warnings if w['gravidade'] == 'MÉDIA'); low = sum(1 for w in all_warnings if w['gravidade'] == 'BAIXA')
            print(f"⚠️ ESTRUTURA SUSPEITA: {len(all_warnings)} anomalia(s) detectada(s).\n   Resumo por Gravidade: Alta={high}, Média={medium}, Baixa={low}")
            if any(w for t in self.tracks if t.media_type == MediaType.AUDIO for w in t.warnings if w['gravidade'] != 'INFO'): print("\n   -> A trilha de áudio apresenta inconsistências estruturais consistentes com manipulação.")
            else: print("\n   -> A trilha de áudio aparenta estar estruturalmente consistente.")
        print()
        return all_warnings


def analisar_arquivo(caminho: str):
    analyzer = STTSAnalyzer(caminho)
    try:
        analyzer.find_boxes()
        warnings = analyzer.generate_report()
        return analyzer, warnings
    except Exception as e:
        print(f"❌ Erro ao analisar arquivo: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if __name__ == "__main__":
    caminho_do_arquivo = './Q/calopsita.mp4' # Substitua pelo caminho do seu arquivo
    print("✨ Analisador de Estrutura MP4/M4A (v3.3 - Detecção de Drift)")
    print("─"*70)
    if Path(caminho_do_arquivo).exists():
        analyzer_obj, warnings_list = analisar_arquivo(caminho_do_arquivo)
    else:
        print(f"ERRO: Arquivo não encontrado em '{caminho_do_arquivo}'")

✨ Analisador de Estrutura MP4/M4A (v3.3 - Detecção de Drift)
──────────────────────────────────────────────────────────────────────
ANÁLISE FORENSE DE ESTRUTURA: calopsita.mp4

 Mídia Geral (Movie): Timescale=10000, Duração=5.21s

────────────────────────────────────────────────────────────────────────────────
🎬 TRILHA 0 (Tipo: Video) - EM ANÁLISE...
────────────────────────────────────────────────────────────────────────────────
  ✅ Nenhuma anomalia estrutural detectada nesta trilha.

────────────────────────────────────────────────────────────────────────────────
🎵 TRILHA 1 (Tipo: Audio) - EM ANÁLISE...
────────────────────────────────────────────────────────────────────────────────
  [Observações Informativas]
    - Padrão de correção de sincronismo (drift) detectado: Variação sutil e repetitiva entre deltas (1023 e 1024) para manter sincronia. É um artefato técnico, não uma edição.
  ✅ Nenhuma anomalia com significado forense detectada nesta trilha.

DIAGNÓSTICO FINAL
✅ ESTRUTURA C

### 2. Função de Execução

A função `analisar_arquivo` atua como um invólucro (wrapper) para orquestrar o processo. Ela instancia a classe `STTSAnalyzer`, inicia a busca pelas caixas (`find_boxes`), gera o relatório e retorna os resultados.

In [51]:
def analisar_arquivo(caminho: str):
    """
    Analisa um arquivo MP4/M4A em busca de rastros de edição
    com análise diferenciada por tipo de mídia (áudio vs vídeo).
    
    Retorna:
        analyzer: Objeto STTSAnalyzer com todas as tracks e dados parseados.
        warnings: Lista de todos os warnings detectados.
    """
    analyzer = STTSAnalyzer(caminho)
    
    try:
        analyzer.find_boxes()
        warnings = analyzer.generate_report()
        
        return analyzer, warnings
        
    except Exception as e:
        print(f"Erro ao analisar arquivo: {e}")
        import traceback
        traceback.print_exc()
        return None, None

### 3. Execução da Análise

Para executar a análise, siga os passos:

1.  Assegure que o arquivo de mídia (`.mp4`, `.m4a`, etc.) esteja no mesmo diretório que este notebook, ou forneça o caminho completo.
2.  Modifique a variável `caminho_do_arquivo` na célula abaixo com o nome do seu arquivo.
3.  Execute a célula para iniciar a análise e visualizar o relatório.

In [60]:
# Modifique o caminho do arquivo na linha abaixo
caminho_do_arquivo = './concatenado.m4a'

print("Analisador de Estrutura MP4/M4A")
print("Análise diferenciada para trilhas de áudio e vídeo.")
print("─"*50)

if Path(caminho_do_arquivo).exists():
    analyzer_obj, warnings_list = analisar_arquivo(caminho_do_arquivo)
else:
    print(f"ERRO: Arquivo não encontrado em '{caminho_do_arquivo}'")
    print("Por favor, verifique o caminho e o nome do arquivo.")

Analisador de Estrutura MP4/M4A
Análise diferenciada para trilhas de áudio e vídeo.
──────────────────────────────────────────────────
ANÁLISE FORENSE DE ESTRUTURA: concatenado.m4a

 Mídia Geral (Movie): Timescale=1000, Duração=21.79s

────────────────────────────────────────────────────────────────────────────────
🎵 TRILHA 0 (Tipo: Audio) - EM ANÁLISE...
────────────────────────────────────────────────────────────────────────────────
  [Anomalias Detectadas]
    🔴 [ALTA] GAP_TEMPORAL:
        Gap temporal significativo na entrada 2
        -> Delta de 38656 é 37.8x maior que o padrão (1024). Consistente com remoção de 0.877s.
    🟡 [MÉDIA] DURACAO_INCONSISTENTE:
        Duração de amostra inconsistente na entrada 4
        -> Delta de 768 é menor que o padrão (1024).

DIAGNÓSTICO FINAL
⚠️ ESTRUTURA SUSPEITA: 2 anomalia(s) detectada(s).
   Resumo por Gravidade: Alta=1, Média=1, Baixa=0

   -> A trilha de áudio apresenta inconsistências estruturais consistentes com manipulação.

